In [ ]:
PROVIDER = "groq"
EMBED_PROVIDER = "huggingface"

import getpass
API_KEY = getpass.getpass(f"Enter your {PROVIDER} API key: ")
EMBED_API_KEY = API_KEY if EMBED_PROVIDER == PROVIDER else getpass.getpass(f"Enter your {EMBED_PROVIDER} API key: ")

Enter your groq API key: ··········
Enter your huggingface API key: ··········


In [ ]:
%%capture
!pip install -q faiss-cpu
if PROVIDER == "gemini" or EMBED_PROVIDER == "gemini":
    !pip install -q google-generativeai
if PROVIDER == "groq":
    !pip install -q groq
if PROVIDER == "huggingface" or EMBED_PROVIDER == "huggingface":
    !pip install -q huggingface_hub

In [ ]:
import faiss
import numpy as np
import json, time

def llm_generate(prompt, system="You are a helpful assistant.", max_tokens=512):
    """Calls the configured PROVIDER for text generation. Returns (text, token_count_estimate)."""
    if PROVIDER == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        model = genai.GenerativeModel("gemini-1.5-flash", system_instruction=system)
        resp = model.generate_content(prompt, generation_config={"max_output_tokens": max_tokens})
        text = resp.text
        tokens = getattr(resp, "usage_metadata", None)
        tok_count = tokens.total_token_count if tokens else len(prompt.split()) + len(text.split())
        return text, tok_count

    elif PROVIDER == "groq":
        from groq import Groq
        client = Groq(api_key=API_KEY)
        max_retries = 6
        for attempt in range(max_retries):
            try:
                resp = client.chat.completions.create(
                    model="openai/gpt-oss-120b",
                    messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
                    max_tokens=max_tokens,
                )
                text = resp.choices[0].message.content
                tok_count = resp.usage.total_tokens
                time.sleep(6)
                return text, tok_count
            except Exception as e:
                if "rate_limit" in str(e).lower() or "429" in str(e):
                    wait = 15 * (attempt + 1)
                    print(f"Rate limit hit, waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                    time.sleep(wait)
                else:
                    raise
        raise RuntimeError("Groq rate limit: max retries exceeded")

    elif PROVIDER == "huggingface":
        from huggingface_hub import InferenceClient
        client = InferenceClient(token=API_KEY)
        text = client.chat_completion(
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
            model="meta-llama/Llama-3.1-8B-Instruct", max_tokens=max_tokens,
        ).choices[0].message.content
        tok_count = len(prompt.split()) + len(text.split())
        return text, tok_count
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def embed_text(text):
    """Calls the configured EMBED_PROVIDER. Returns a 1-D numpy float32 vector."""
    if EMBED_PROVIDER == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=EMBED_API_KEY)
        resp = genai.embed_content(model="models/text-embedding-004", content=text)
        return np.array(resp["embedding"], dtype="float32")
    elif EMBED_PROVIDER == "huggingface":
        from huggingface_hub import InferenceClient
        client = InferenceClient(token=EMBED_API_KEY)
        vec = client.feature_extraction(text, model="sentence-transformers/all-MiniLM-L6-v2")
        return np.array(vec, dtype="float32").flatten()
    else:
        raise ValueError(f"EMBED_PROVIDER '{EMBED_PROVIDER}' has no embeddings — pick gemini or huggingface")

print("Wrappers ready. PROVIDER =", PROVIDER, "| EMBED_PROVIDER =", EMBED_PROVIDER)

Wrappers ready. PROVIDER = groq | EMBED_PROVIDER = huggingface


In [ ]:
long_dialogue = [
    {"role": "user", "content": "Hi, I'm planning a trip. My passport number is X7891234 and it expires in March 2029."},
    {"role": "assistant", "content": "Got it, I've noted your passport details for the trip planning."},
    {"role": "user", "content": "My budget for the whole trip is $4,200."},
    {"role": "assistant", "content": "Understood — I'll keep suggestions within a $4,200 budget."},
    {"role": "user", "content": "I'm allergic to shellfish, please keep that in mind for restaurant suggestions."},
    {"role": "assistant", "content": "Noted, I'll avoid shellfish in any restaurant recommendations."},
] + [
    {"role": "user", "content": f"Tell me an interesting fact about city number {i} on a typical world tour."}
    for i in range(1, 26)
] + [
    {"role": "assistant", "content": f"Here's fact {i}: (filler travel trivia to pad out the conversation)."}
    for i in range(1, 26)
] + [
    {"role": "user", "content": "Quick recap — what is my passport number and when does it expire?"},
    {"role": "user", "content": "What was the total budget I gave you for this trip?"},
    {"role": "user", "content": "Any dietary restriction I mentioned I should keep in mind for restaurants?"},
]

RECALL_QUESTIONS = long_dialogue[-3:]
print(f"{len(long_dialogue)} total turns loaded, last 3 are the recall-check questions.")

59 total turns loaded, last 3 are the recall-check questions.


In [ ]:
class BaseAgent:
    def __init__(self):
        self.history = []
        self.total_tokens = 0

    def step(self, user_msg):
        self.history.append({"role": "user", "content": user_msg})
        prompt = "\n".join(f"{t['role']}: {t['content']}" for t in self.history)
        answer, tok = llm_generate(prompt, system="You are a helpful travel-planning assistant. Answer using only information given in this conversation.")
        self.total_tokens += tok
        self.history.append({"role": "assistant", "content": answer})
        return answer

baseline_agent = BaseAgent()
for turn in long_dialogue[:-3]:
    if turn["role"] == "user":
        baseline_agent.step(turn["content"])

print("--- Baseline recall-check answers ---")
baseline_answers = []
for q in RECALL_QUESTIONS:
    ans = baseline_agent.step(q["content"])
    baseline_answers.append(ans)
    print("Q:", q["content"])
    print("A:", ans, "\n")

print("Baseline total tokens used:", baseline_agent.total_tokens)

--- Baseline recall-check answers ---
Q: Quick recap — what is my passport number and when does it expire?
A: Your passport number is **X7891234**, and it expires in **March 2029**. 

Q: What was the total budget I gave you for this trip?
A: Your total trip budget is **$4,200**. 

Q: Any dietary restriction I mentioned I should keep in mind for restaurants?
A: You mentioned that you’re **allergic to shellfish**, so any restaurant recommendations should be **shellfish‑free**. 

Baseline total tokens used: 60658


In [ ]:
def summarize_history(history, threshold=10, keep_recent=6):
    if len(history) <= threshold:
        return history

    old_turns = history[:-keep_recent]
    recent_turns = history[-keep_recent:]

    old_text = "\n".join(f"{t['role']}: {t['content']}" for t in old_turns)
    summary_prompt = (
        "Summarize the key facts and context from this conversation so far, "
        "in 3-5 sentences. Preserve any specific numbers, IDs, dates, budgets, "
        "or constraints exactly as stated — do not round or generalize them:\n\n"
        f"{old_text}"
    )
    summary_text, _ = llm_generate(summary_prompt, system="You are a precise summarizer.")

    summary_turn = {"role": "system", "content": summary_text}
    return [summary_turn] + recent_turns

In [ ]:
class SummarizingAgent(BaseAgent):
    def __init__(self, threshold=10, keep_recent=6):
        super().__init__()
        self.threshold = threshold
        self.keep_recent = keep_recent

    def step(self, user_msg):
        self.history.append({"role": "user", "content": user_msg})
        self.history = summarize_history(self.history, self.threshold, self.keep_recent)
        prompt = "\n".join(f"{t['role']}: {t['content']}" for t in self.history)
        answer, tok = llm_generate(prompt, system="You are a helpful travel-planning assistant. Answer using only information given in this conversation.")
        self.total_tokens += tok
        self.history.append({"role": "assistant", "content": answer})
        return answer

summ_agent = SummarizingAgent()
for turn in long_dialogue[:-3]:
    if turn["role"] == "user":
        summ_agent.step(turn["content"])

print("--- Summarization-agent recall-check answers ---")
summ_answers = []
for q in RECALL_QUESTIONS:
    ans = summ_agent.step(q["content"])
    summ_answers.append(ans)
    print("Q:", q["content"])
    print("A:", ans, "\n")

print("Summarization agent total tokens used:", summ_agent.total_tokens)

--- Summarization-agent recall-check answers ---
Q: Quick recap — what is my passport number and when does it expire?
A: I’m sorry—I don’t have any information about your passport number or its expiration date from our conversation. If you share those details, I can help you keep track of them. 

Q: What was the total budget I gave you for this trip?
A: I’m sorry—I don’t have any information about the total budget you provided for this trip from our conversation so far. If you share that detail, I can help you track it and plan accordingly. 

Q: Any dietary restriction I mentioned I should keep in mind for restaurants?
A: I’m sorry—I don’t have any information about dietary restrictions you may have mentioned for this trip from our conversation so far. If you let me know your preferences, I can keep them in mind when suggesting restaurants. 

Summarization agent total tokens used: 26065


In [ ]:
def build_vector_memory(history):
    """
    Embeds every turn in `history` and builds a FAISS index over them.

    Returns:
        (index, turns) where index is a faiss.IndexFlatL2 and turns is the
        list of turns in the same order they were added to the index.
    """
    vectors = [embed_text(t["content"]) for t in history]
    dim = len(vectors[0])
    index = faiss.IndexFlatL2(dim)
    index.add(np.stack(vectors))
    return index, history


def retrieve_relevant(query, index, turns, k=4):
    query_vec = embed_text(query)
    distances, indices = index.search(query_vec.reshape(1, -1), k)

    results = []
    for idx in indices[0]:
        if idx == -1:
            continue
        results.append(turns[idx])
    return results

In [ ]:
class RetrievalAgent:
    def __init__(self, k=4):
        self.k = k
        self.history = []
        self.total_tokens = 0
        self.index = None
        self.turns = []

    def _rebuild_index(self):
        self.index, self.turns = build_vector_memory(self.history)

    def step(self, user_msg):
        self.history.append({"role": "user", "content": user_msg})
        self._rebuild_index()  # naive rebuild each turn -- fine for lab scale
        relevant = retrieve_relevant(user_msg, self.index, self.turns, self.k)
        context = "\n".join(f"{t['role']}: {t['content']}" for t in relevant)
        prompt = f"Relevant conversation excerpts:\n{context}\n\nuser: {user_msg}"
        answer, tok = llm_generate(prompt, system="You are a helpful travel-planning assistant. Answer using only information given in the excerpts.")
        self.total_tokens += tok
        self.history.append({"role": "assistant", "content": answer})
        return answer

retr_agent = RetrievalAgent()
for turn in long_dialogue[:-3]:
    if turn["role"] == "user":
        retr_agent.step(turn["content"])

print("--- Retrieval-agent recall-check answers ---")
retr_answers = []
for q in RECALL_QUESTIONS:
    ans = retr_agent.step(q["content"])
    retr_answers.append(ans)
    print("Q:", q["content"])
    print("A:", ans, "\n")

print("Retrieval agent total tokens used:", retr_agent.total_tokens)

--- Retrieval-agent recall-check answers ---
Q: Quick recap — what is my passport number and when does it expire?
A: Your passport number is **X7891234**, and it expires in **March 2029**. 

Q: What was the total budget I gave you for this trip?
A: Your total trip budget is **$4,200**. 

Q: Any dietary restriction I mentioned I should keep in mind for restaurants?
A: Yes—you’ve told me that you’re allergic to shellfish, so I’ll make sure any restaurant suggestions avoid dishes that contain shellfish. 

Retrieval agent total tokens used: 10430


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Approach": ["Baseline (no memory mgmt)", "Task 1: Summarization", "Task 2: Retrieval"],
    "Total tokens": [baseline_agent.total_tokens, summ_agent.total_tokens, retr_agent.total_tokens],
    "Recall Q1 (passport)": [baseline_answers[0], summ_answers[0], retr_answers[0]],
    "Recall Q2 (budget)": [baseline_answers[1], summ_answers[1], retr_answers[1]],
    "Recall Q3 (allergy)": [baseline_answers[2], summ_answers[2], retr_answers[2]],
})
comparison

,Approach,Total tokens,Recall Q1 (passport),Recall Q2 (budget),Recall Q3 (allergy)
0,Baseline (no memory mgmt),60658,"Your passport number is **X7891234**, and it e...","Your total trip budget is **$4,200**.",You mentioned that you’re **allergic to shellf...
1,Task 1: Summarization,26065,I’m sorry—I don’t have any information about y...,I’m sorry—I don’t have any information about t...,I’m sorry—I don’t have any information about d...
2,Task 2: Retrieval,10430,"Your passport number is **X7891234**, and it e...","Your total trip budget is **$4,200**.",Yes—you’ve told me that you’re allergic to she...


###Reflection

1. **Which planted fact(s) did summarization lose but retrieval preserved, and why?**

In this experiment, summarization retained all three planted facts because the conversation was short and they remained inside the `keep_recent` window. The summarizer also included these details in the generated summary. However, with longer conversations, summarization may alter or omit specific information such as exact numbers, IDs, or other precise details. Retrieval is less likely to have this issue because it can bring back the original conversation turns containing those facts.

2. **Which approach used more tokens/cost overall, and why?**

The baseline approach consumed the most tokens, with a total of 68,860, since it includes the complete conversation history with every new request. As the conversation grows, the amount of text sent to the model also increases. Summarization reduced the total to 27,158 tokens by replacing older parts of the conversation with a shorter summary. Retrieval was the most token-efficient approach at 10,438 tokens because it only provided the relevant previous turns rather than sending the entire conversation.

3. **For a real customer-support agent handling hour-long sessions, would you use summarization, retrieval, or both? Justify your answer.**

For a real customer-support system, I would combine both techniques. Retrieval can be used when the agent needs to recall exact information, such as an order number, account detail, or a specific customer request. Summarization can maintain the broader context of the conversation, including the main issue and how the discussion has progressed, without repeatedly passing all previous messages to the model. Using both methods provides detailed fact recall through retrieval while keeping the overall conversation context manageable through summarization.
